In [1]:
import numpy as np
import pathlib as P
import pickle
import pandas as pd

In [2]:
ns = ["cc", "mf", "bp"]
ontology = ["cellular_component", "molecular_function", "biological_process"]

In [3]:
def align_labels(predicted_results, o1, o2, fill_value=0.0):
    """
    Align predicted results to match new label order, handling missing labels.
    
    Args:
        predicted_results: numpy array of shape (N, len(o1))
        o1: list - original label order
        o2: list - desired label order (can have different labels/length)
        fill_value: value to use for labels in o2 that don't exist in o1
    
    Returns:
        numpy array of shape (N, len(o2)) - aligned predictions
    """
    N = predicted_results.shape[0]
    M_new = len(o2)
    
    # Initialize new result array
    aligned_results = np.full((N, M_new), fill_value, dtype=predicted_results.dtype)
    
    # Create mapping from label to column index in o1
    o1_label_to_idx = {label: idx for idx, label in enumerate(o1)}
    
    # Fill the aligned results
    for new_idx, label in enumerate(o2):
        if label in o1_label_to_idx:
            old_idx = o1_label_to_idx[label]
            aligned_results[:, new_idx] = predicted_results[:, old_idx]
        # If label not in o1, keep the fill_value (already set)
    
    return aligned_results

In [4]:
# previous order of labels
lable_path = [f"/data0/shaojiangyi/pprogo-flg-1/lmj_results/results/fused_three_models_esm2_v3_{x}_optimized/union_go_terms.npy" for x in ns]
prep_labels = [np.load(p).tolist() for p in lable_path]

In [5]:
# current order of labels
lable_path = [f"/data0/shaojiangyi/pprogo-flg-1/lmj_results/results/fused_three_models_esm2_v3_{x}_optimized/union_go_terms.npy" for x in ns]
curr_labels = [np.load(p).tolist() for p in lable_path]

In [6]:
[print(len(l)) for l in curr_labels]
[print(len(l)) for l in prep_labels]

2881
6860
21822
2881
6860
21822


[None, None, None]

In [ ]:
root_paths = [
    "/data0/shaojiangyi/pprogo-flg-1/results/fused_three_models_esm2_v3_{}_optimized",
]
savinng_paths = [
    P.Path("/data0/shaojiangyi/pprogo-flg-1/results/fused_three_models_esm2_optimized_aligned"),
]
for i, root_path in enumerate(root_paths):
    # pred_paths = [P.Path(root_path) / f"{ns[i]}_prop_pred_result.npy" for i in range(len(ns))]
    pred_paths = [(P.Path(root_path.join(x)) / "fused_labels_union.npy",
                   P.Path(root_path.join(x)) / "fused_preds_union.npy") for x in ns]
    preds = [np.stack((np.load(p[0]), np.load(p[1]))) for p in pred_paths]
    # Align predictions to the new label order
    aligned_preds = [np.stack((align_labels(pred[0], prep_labels[i], curr_labels[i]),
                               align_labels(pred[1], prep_labels[i], curr_labels[i]))) 
                     for i, pred in enumerate(preds)]
    # print shape
    print([p.shape for p in aligned_preds])
    # Save aligned predictions
    for i, pred in enumerate(aligned_preds):
        np.save(savinng_paths[i] / f"{ns[i]}_result_aligned.npy", pred)

[(2, 268, 2881), (2, 505, 6860), (2, 491, 21822)]
